# Late-arriving data and replay

Apply delayed records and corrections by business key, preserve event time, and prove that replay does not duplicate or regress data.

![Late-arriving data and replay](assets/05_flow.png)


## Connect the notebook to Spark

Run from top to bottom in a fresh Python kernel. Close the previous Spark session first.
Connection values use the environment when present, otherwise the Kubernetes Service names below.
Credentials stay in the existing course environment. No extra workbench manifest update is needed.
Short Service names resolve within the Pod's namespace; use a fully qualified Service name when crossing namespaces.
A Service name stays stable when its Pods are replaced. A container name or Pod IP is not the connection contract.

The notebook kernel is the **driver**. `.master(master_url)` connects this application to the
standalone Spark master. The master allocates executors on its workers. Workers must also reach
the notebook driver address and its ports. The Spark services and workbench run inside Kubernetes;
this is a standalone Spark application, rather than native Kubernetes submission.

No separate `spark-submit` command is needed while running notebook cells. Read the master URL
from the environment and inspect `spark.sparkContext.master` after the connection is created.


In [ ]:
# Import the standard Python and Spark classes.
import os
import re
import socket
import uuid

from datetime import date, datetime
from decimal import Decimal

from pyspark.sql import SparkSession


In [ ]:
# Use environment overrides or the stable Kubernetes Service names.
master_url = os.environ.get('SPARK_MASTER_URL', 'spark://spark-master:7077')
driver_host = os.environ.get('SPARK_DRIVER_HOST', 'jupyter')

hdfs_uri = os.environ.get('HDFS_URI', 'hdfs://namenode:9000').rstrip('/')
metastore_uri = os.environ.get('HIVE_METASTORE_URI', 'thrift://hive-metastore:9083')

polaris_base_url = os.environ.get('POLARIS_BASE_URL', 'http://polaris:8181').rstrip('/')
polaris_uri = os.environ.get('POLARIS_URI', polaris_base_url + '/api/catalog').rstrip('/')
polaris_warehouse = os.environ.get('POLARIS_WAREHOUSE', os.environ.get('POLARIS_CATALOG', 'training'))
s3_endpoint = os.environ.get('S3_ENDPOINT', 'http://seaweedfs:8333')
storage_region = os.environ.get('AWS_DEFAULT_REGION', os.environ.get('AWS_REGION', 'us-east-1'))
polaris_realm = os.environ.get('POLARIS_REALM', 'POLARIS')

assert master_url.startswith('spark://'), 'Use the course standalone Spark master URL'


In [ ]:
# Display the resolved connection values before Spark starts.
resolved_connections = {
    'Spark master': master_url,
    'Notebook driver': driver_host,
    'HDFS filesystem': hdfs_uri,
    'Hive metastore': metastore_uri,
    'Polaris REST catalog': polaris_uri,
    'Polaris catalog name': polaris_warehouse,
    'Polaris realm': polaris_realm,
    'S3 endpoint': s3_endpoint,
    'Storage region': storage_region,
}

for setting, value in resolved_connections.items():
    print(f'{setting:24} = {value}')


In [ ]:
# Read existing credentials without printing their values.
polaris_credential = os.environ.get('POLARIS_CREDENTIAL')
if not polaris_credential:
    polaris_credential = (
        os.environ['POLARIS_CLIENT_ID'] + ':' + os.environ['POLARIS_CLIENT_SECRET']
    )

s3_access_key = os.environ['AWS_ACCESS_KEY_ID']
s3_secret_key = os.environ['AWS_SECRET_ACCESS_KEY']


### Give this execution its own tables and files

`ns` is the database or namespace used by the table names below. `root` is this execution's HDFS directory.
A new run ID keeps a full rerun separate from earlier work. Keep the printed values if you want to inspect the results later.


In [ ]:
# Choose isolated names for this notebook execution.
run_id = os.environ.get('LESSON_RUN_ID', uuid.uuid4().hex[:10])
assert re.fullmatch(r'[a-z0-9_]{1,32}', run_id), 'Use a lowercase execution ID'

ns = f'migration11_05_{run_id}'
root = hdfs_uri + f'/workshop/11-iceberg-migration/05/{run_id}'

print('Namespace:', ns)
print('HDFS directory:', root)


In [ ]:
# Make the warehouse locations and Polaris catalog selection explicit.
hive_warehouse = root + '/managed'
hadoop_iceberg_warehouse = root + '/iceberg-warehouse'
hive_iceberg_warehouse = root + '/iceberg-hive-warehouse'

print('Hive managed warehouse:      ', hive_warehouse)
print('HadoopCatalog warehouse:     ', hadoop_iceberg_warehouse)
print('Hive Iceberg warehouse:      ', hive_iceberg_warehouse)
print('Polaris warehouse parameter: ', polaris_warehouse, '(catalog name, not an S3 path)')


### Configure the driver and executors

`.master(master_url)` selects the cluster. The driver address is different: executors use it to call back to this notebook.
Driver ports must be reachable from the worker network. A small executor allocation is sufficient for these datasets.


In [ ]:
# Connect the application to the master and advertise the notebook driver.
builder = (
    SparkSession.builder
    .appName('iceberg-migration-05')
    .master(master_url)
    .config('spark.driver.host', driver_host)
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.driver.port', '7078')
    .config('spark.blockManager.port', '7079')
)


In [ ]:
# Set the resource limits, time zone and credential redaction.
builder = (
    builder
    .config('spark.cores.max', '1')
    .config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.sql.adaptive.enabled', 'false')
    .config('spark.sql.session.timeZone', 'UTC')
    .config('spark.openlineage.disabled', 'true')
    .config('spark.redaction.regex', '(?i)secret|password|token|access[.]?key|credential')
)


### Configure the catalogs

`spark_catalog` reads Hive tables and supports in-place conversion through the metastore.
`lake` is a HadoopCatalog: its catalog metadata and data live on HDFS, without the Hive metastore.
`iceberg` is a HiveCatalog: table registration lives in the Hive metastore, with files on HDFS.
`polaris` is a REST catalog for Iceberg tables in the object-store warehouse.
Catalog configuration is required before creating the Spark session.


In [ ]:
# Configure the default HDFS filesystem and managed-table warehouse.
builder = (
    builder
    .config('spark.hadoop.fs.defaultFS', hdfs_uri)
    .config('spark.hadoop.dfs.client.use.datanode.hostname', 'true')
    .config('spark.sql.warehouse.dir', hive_warehouse)
)


In [ ]:
# Enable Iceberg SQL and keep the Hive metastore available.
builder = (
    builder
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .config('spark.hadoop.hive.metastore.uris', metastore_uri)
    .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')
    .config('spark.sql.catalog.spark_catalog.type', 'hive')
    .config('spark.sql.catalog.spark_catalog.uri', metastore_uri)
    .config('spark.sql.catalog.spark_catalog.warehouse', hive_warehouse)
)


In [ ]:
# Keep the HDFS Iceberg warehouse under this execution directory.
builder = (
    builder
    .config('spark.sql.catalog.lake', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.lake.type', 'hadoop')
    .config('spark.sql.catalog.lake.warehouse', hadoop_iceberg_warehouse)
)


In [ ]:
# Register a Hive-metastore Iceberg catalog with HDFS storage.
builder = (
    builder
    .config('spark.sql.catalog.iceberg', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.iceberg.type', 'hive')
    .config('spark.sql.catalog.iceberg.uri', metastore_uri)
    .config('spark.sql.catalog.iceberg.warehouse', hive_iceberg_warehouse)
)


### Configure both S3 clients

Spark reads and writes ordinary `s3a://` files through Hadoop S3A.
Iceberg's Polaris tables use `s3://` locations through S3FileIO. These are separate clients;
configure the endpoint, credentials and path-style access for both.
HTTP disables S3A TLS only for this course endpoint. HTTPS keeps TLS enabled.


In [ ]:
# Configure Hadoop S3A for ordinary Spark object-store files.
builder = (
    builder
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.endpoint', s3_endpoint)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', str(s3_endpoint.startswith('https://')).lower())
    .config('spark.hadoop.fs.s3a.aws.credentials.provider', 'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.hadoop.fs.s3a.access.key', s3_access_key)
    .config('spark.hadoop.fs.s3a.secret.key', s3_secret_key)
)


In [ ]:
# Connect the Polaris REST catalog using credentials from the environment.
builder = (
    builder
    .config('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type', 'rest')
    .config('spark.sql.catalog.polaris.uri', polaris_uri)
    .config('spark.sql.catalog.polaris.warehouse', polaris_warehouse)
    .config('spark.sql.catalog.polaris.oauth2-server-uri', polaris_uri + '/v1/oauth/tokens')
    .config('spark.sql.catalog.polaris.credential', polaris_credential)
    .config('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.polaris.header.Polaris-Realm', polaris_realm)
    .config('spark.sql.catalog.polaris.rest-metrics-reporting-enabled', 'false')
)


In [ ]:
# Let Iceberg read and write the configured S3 warehouse.
builder = (
    builder
    .config('spark.sql.catalog.polaris.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.polaris.s3.endpoint', s3_endpoint)
    .config('spark.sql.catalog.polaris.s3.path-style-access', 'true')
    .config('spark.sql.catalog.polaris.s3.access-key-id', s3_access_key)
    .config('spark.sql.catalog.polaris.s3.secret-access-key', s3_secret_key)
    .config('spark.sql.catalog.polaris.client.region', storage_region)
)


In [ ]:
# Start Spark and display the master actually used.
spark = builder.enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

print('Spark version:', spark.version)
print('Connected master:', spark.sparkContext.master)

assert spark.sparkContext.master == master_url, 'Spark must use the configured master'


### Verify that work reaches an executor

The following small task runs on a worker. Its process hostname must differ from the notebook driver's hostname.
This checks that the application is using a separate executor instead of silently running in local mode.


In [ ]:
# Verify distributed execution before writing course data.
worker_names = (
    spark.sparkContext
    .parallelize([1], 1)
    .map(lambda _: __import__('socket').gethostname())
    .collect()
)

assert all(name != socket.gethostname() for name in worker_names), 'Use a separate executor'
print('PASS: Distributed Spark executor verified')


In [ ]:
# Put managed Hive data under this execution directory.
spark.sql(f"""
    CREATE DATABASE IF NOT EXISTS spark_catalog.{ns}
    LOCATION '{root}/managed'
""")


In [ ]:
# Create the namespace for HDFS Iceberg exercises.
spark.sql(f"""
    CREATE NAMESPACE IF NOT EXISTS lake.{ns}
""")


In [ ]:
# Use the existing Hive database through the named Iceberg catalog.
spark.sql(f"""
    CREATE NAMESPACE IF NOT EXISTS iceberg.{ns}
""")


In [ ]:
# Create the namespace for object-store Iceberg exercises.
spark.sql(f"""
    CREATE NAMESPACE IF NOT EXISTS polaris.{ns}
""")


## Stage the source data in HDFS
These six orders span three days. Amounts use fixed precision and all timestamps use UTC.
The explicit `order_date` column represents a traditional Hive partition key.


In [ ]:
# Define the staged rows and their explicit column types.
staged_orders_df = spark.createDataFrame(
    [
        (101, 1, datetime(2025, 1, 1, 9), Decimal('120.00'), 'PLACED', datetime(2025, 1, 1, 9), date(2025, 1, 1)),
        (102, 2, datetime(2025, 1, 1, 11), Decimal('80.00'), 'PLACED', datetime(2025, 1, 1, 11),
            date(2025, 1, 1)),
        (103, 1, datetime(2025, 1, 2, 10), Decimal('200.00'), 'PAID', datetime(2025, 1, 2, 12), date(2025, 1, 2)),
        (104, 3, datetime(2025, 1, 2, 15), Decimal('50.00'), 'PLACED', datetime(2025, 1, 2, 15),
            date(2025, 1, 2)),
        (105, 2, datetime(2025, 1, 3, 8), Decimal('300.00'), 'PAID', datetime(2025, 1, 3, 10), date(2025, 1, 3)),
        (106, 4, datetime(2025, 1, 3, 18), Decimal('150.00'), 'PLACED', datetime(2025, 1, 3, 18),
            date(2025, 1, 3)),
    ],
    """
        order_id INT,
        customer_id INT,
        order_ts TIMESTAMP,
        amount DECIMAL(12,2),
        status STRING,
        updated_at TIMESTAMP,
        order_date DATE
    """,
)


In [ ]:
# Choose the partitioned source directory.
source_path = root + '/orders/partitioned'

staged_orders_df.write.mode('errorifexists').partitionBy('order_date').parquet(source_path)


In [ ]:
# Make these rows available to Spark SQL.
staged_orders_df.createOrReplaceTempView('staged_orders')


In [ ]:
# Register the external source files.
spark.sql(f"""
    CREATE EXTERNAL TABLE spark_catalog.{ns}.orders (
    order_id INT, customer_id INT, order_ts TIMESTAMP, amount DECIMAL(12,2),
    status STRING, updated_at TIMESTAMP
    )
    PARTITIONED BY (order_date DATE)
    STORED AS PARQUET
    LOCATION '{source_path}'
""")


In [ ]:
# Register the source partition directories.
spark.sql(f"""
    MSCK REPAIR TABLE spark_catalog.{ns}.orders
""")


In [ ]:
# Six source orders registered.
assert spark.table(f'spark_catalog.{ns}.orders').count() == 6, 'Six source orders registered'

print('PASS: Six source orders registered')


In [ ]:
# Create the table for this step.
spark.sql(f"""
    CREATE TABLE polaris.{ns}.orders
    USING iceberg
    PARTITIONED BY (days(order_ts))
    TBLPROPERTIES ('format-version'='2')
    AS SELECT *
    FROM spark_catalog.{ns}.orders
""").show(truncate=False)


## Stage a delayed batch
Order 107 arrives on January 5 but belongs to January 1. Order 102 is corrected and order 104 is cancelled.
The batch also includes an older version of order 102 and an exact duplicate of order 107.
`updated_at` is the source change version; `order_ts` is the business event time.


In [ ]:
# Define the staged rows and their explicit column types.
staged_order_changes_df = spark.createDataFrame(
    [
        (107, 5, datetime(2025, 1, 1, 16), Decimal('75.00'), 'PLACED', datetime(2025, 1, 5, 9),
            date(2025, 1, 1), 'batch_002'),
        (107, 5, datetime(2025, 1, 1, 16), Decimal('75.00'), 'PLACED', datetime(2025, 1, 5, 9),
            date(2025, 1, 1), 'batch_002'),
        (102, 2, datetime(2025, 1, 1, 11), Decimal('90.00'), 'PAID', datetime(2025, 1, 5, 10),
            date(2025, 1, 1), 'batch_002'),
        (102, 2, datetime(2025, 1, 1, 11), Decimal('85.00'), 'PLACED', datetime(2025, 1, 4, 10),
            date(2025, 1, 1), 'batch_002'),
        (104, 3, datetime(2025, 1, 2, 15), Decimal('50.00'), 'CANCELLED', datetime(2025, 1, 5, 11),
            date(2025, 1, 2), 'batch_002'),
    ],
    """
        order_id INT,
        customer_id INT,
        order_ts TIMESTAMP,
        amount DECIMAL(12,2),
        status STRING,
        updated_at TIMESTAMP,
        order_date DATE,
        batch_id STRING
    """,
)


In [ ]:
# Choose the HDFS directory for this incoming batch.
changes_path = root + '/order-changes/batch_002'

staged_order_changes_df.write.mode('errorifexists').parquet(changes_path)


In [ ]:
# Make these rows available to Spark SQL.
spark.read.parquet(changes_path).createOrReplaceTempView('raw_changes')


## Reject conflicting versions before MERGE
An exact duplicate is safe to remove. Two different payloads with the same key and version need a source rule or quarantine.
Do not choose one arbitrarily. Deduplicate before MERGE so one target row never matches multiple source rows.


In [ ]:
# Expose this SQL result as a temporary view.
spark.sql("""
    CREATE OR REPLACE TEMP VIEW distinct_changes
    AS SELECT DISTINCT *
    FROM raw_changes
""")


In [ ]:
# Read the rows or metadata needed for this check.
conflicts = spark.sql("""
    SELECT order_id, updated_at
    FROM distinct_changes
    GROUP BY order_id, updated_at
    HAVING count(*) > 1
""")


In [ ]:
# No conflicting payloads share a key and version.
assert conflicts.count() == 0, 'No conflicting payloads share a key and version'

print('PASS: No conflicting payloads share a key and version')


In [ ]:
# Expose this SQL result as a temporary view.
spark.sql("""
    CREATE OR REPLACE TEMP VIEW latest_changes AS
    SELECT order_id, customer_id, order_ts, amount, status, updated_at, order_date
    FROM (SELECT *, row_number() OVER (PARTITION BY order_id
    ORDER BY updated_at DESC) AS rn
    FROM distinct_changes)
    WHERE rn = 1
""")


In [ ]:
# Three distinct business keys remain.
assert spark.table('latest_changes').count() == 3, 'Three distinct business keys remain'

print('PASS: Three distinct business keys remain')


In [ ]:
# Keep this value for the next comparison.
merge_sql = f"""
    MERGE INTO polaris.{ns}.orders t
    USING latest_changes s
    ON t.order_id = s.order_id
    WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
"""


In [ ]:
# Run this SQL statement and inspect its result.
spark.sql(merge_sql)


In [ ]:
# Late order inserted exactly once.
assert spark.table(f'polaris.{ns}.orders').count() == 7, 'Late order inserted exactly once'

print('PASS: Late order inserted exactly once')


In [ ]:
# Corrected total is 985.00.
assert spark.sql(f"""
    SELECT sum(amount)
    FROM polaris.{ns}.orders
""").first()[0] == Decimal('985.00'), 'Corrected total is 985.00'

print('PASS: Corrected total is 985.00')


In [ ]:
# Cancellation retained as business state.
assert spark.sql(f"""
    SELECT status
    FROM polaris.{ns}.orders
    WHERE order_id=104
""").first()[0] == 'CANCELLED', 'Cancellation retained as business state'

print('PASS: Cancellation retained as business state')


In [ ]:
# Late order belongs to the original business date.
assert spark.sql(f"""
    SELECT order_date
    FROM polaris.{ns}.orders
    WHERE order_id=107
""").first()[0] == date(2025, 1, 1), 'Late order belongs to the original business date'

print('PASS: Late order belongs to the original business date')


In [ ]:
# Read the rows or metadata needed for this check.
spark.sql(f"""
    SELECT partition, record_count
    FROM polaris.{ns}.orders.partitions
""").show(truncate=False)


In [ ]:
# Keep the result for the comparison below.
before_replay = spark.table(f'polaris.{ns}.orders').collect()


In [ ]:
# Run this SQL statement and inspect its result.
spark.sql(merge_sql)


In [ ]:
# Replaying the batch leaves every row unchanged.
assert set(before_replay) == set(spark.table(f'polaris.{ns}.orders').collect()), 'Replaying the batch leaves every row unchanged'

print('PASS: Replaying the batch leaves every row unchanged')


In [ ]:
# Apply the source rows by business key.
spark.sql(f"""
    MERGE INTO polaris.{ns}.orders t
    USING spark_catalog.{ns}.orders s
    ON t.order_id = s.order_id
    WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")


In [ ]:
# Replaying old history cannot overwrite the correction.
assert spark.sql(f"""
    SELECT amount
    FROM polaris.{ns}.orders
    WHERE order_id=102
""").first()[0] == Decimal('90.00'), 'Replaying old history cannot overwrite the correction'

print('PASS: Replaying old history cannot overwrite the correction')


## Track the batch after data validation
Keep source batch identity, expected key count and validation status. This audit record is useful for operations;
it is not a transaction spanning two Iceberg tables. The MERGE itself must remain replay-safe if the process stops before audit commit.


In [ ]:
# Create the table for this step.
spark.sql(f"""
    CREATE TABLE polaris.{ns}.batch_audit (batch_id STRING, business_keys BIGINT, state STRING)
    USING iceberg
""").show(truncate=False)


In [ ]:
# Write this batch into the destination table.
spark.sql(f"""
    INSERT INTO polaris.{ns}.batch_audit
    VALUES ('batch_002', 3, 'VALIDATED')
""").show(truncate=False)


## Classroom exercises and assignment

1. Explain why partition overwrite would be risky for this delayed batch.
2. Check the cancelled amount separately from the active-order total.
3. Assignment: stage a corrected invoice with a newer version, an old replay and a conflicting equal-version payload. Define the quarantine rule before applying MERGE.


<details>
    <summary>Review points</summary>

    A partial old-date batch must not replace the whole partition. Business state and physical DELETE have different meanings. Timestamp ordering is valid only if the source guarantees its version semantics; otherwise use a durable change sequence. Equal-version conflicts must not be resolved by arbitrary row ordering.

</details>


In [ ]:
# Display the checked result.
print('ALL CORE CHECKS PASSED')


In [ ]:
# Release this notebook Spark session.
spark.stop()
